# 🧪 Tier-2: Power Transformer Health Index & DGA Diagnostics
### Siemens IOT-01 — Predictive Failure Detection for Smart Grid Substations
**Author:** Sovereign Predictive Maintenance Team | **Target Model:** Dissolved Gas Analysis (DGA) & Thermal Degradation Engine  
**Dataset Reference:** `shashwatwork/failure-analysis-in-power-transformers-dataset`

---

## 🎯 Executive Overview & Equipment Context
Power transformers are the most critical and expensive assets in a substation:
* **Monitored Assets:**
  - **Main Transmission Step-Down Transformers** (Siemens Sensformer 400/220 kV, 315 MVA)
  - **On-Load Tap Changers (OLTC)**
  - **High-Voltage Oil-Impregnated Paper Bushings**
* **Primary Failure Modes:**
  - **PD:** Partial Discharge (insulation micro-voids, moisture)
  - **T1/T2:** Low/Medium Thermal Fault ($<700^\circ\text{C}$ winding hotspots)
  - **T3:** Severe High-Temperature Thermal Fault ($>700^\circ\text{C}$ core overheating)
  - **D1/D2:** High-Energy Electrical Arcing (flashover, breakdown)
  - **Aging:** Dielectric loss & paper cellulose degradation

---

## 📊 Notebook Workflow
1. **Environment Setup & Data Ingestion (IEEE C57.104 Schema)**
2. **Standard Diagnostic Ratios (Duval Triangle, Rogers Ratios, TCG)**
3. **Multi-Class Fault Classifier (XGBoost / CatBoost)**
4. **Continuous Asset Health Index Regressor (LightGBM)**
5. **Evaluation: Multi-Class Confusion Matrix & Regression MAE**
6. **Chemical Explainability & Gas Attribution (SHAP)**
7. **Artifact Export for Substation Production API**

In [2]:
# 1. Environment Setup & Core Libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor

import xgboost as xgb
import lightgbm as lgb
import shap

np.random.seed(42)
print("Transformer Diagnostics environment ready.")

Transformer Diagnostics environment ready.


## 2. Ingestion & IEEE C57.104 Compliant Data Generation
Features include key Dissolved Gases (ppm), Oil Breakdown Voltage (BDV), Moisture, and Temperatures.

In [4]:
DATASET_PATH = "data/transformer_failures.csv"
os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)

def load_or_generate_transformer_dataset(path: str, n_samples: int = 1200) -> pd.DataFrame:
    if os.path.exists(path):
        print(f"Loading transformer dataset from: {path}")
        return pd.read_csv(path)
    
    print(f"Transformer dataset not found. Generating IEEE C57.104 / IEC 60599 dataset ({n_samples} samples)...")
    np.random.seed(42)
    
    # Fault classes: 0: Normal, 1: Partial Discharge (PD), 2: Thermal T1/T2, 3: Thermal T3, 4: Electrical Arcing (D1/D2)
    fault_classes = np.random.choice([0, 1, 2, 3, 4], size=n_samples, p=[0.55, 0.12, 0.15, 0.10, 0.08])
    
    # Base nominal gas levels (ppm)
    h2 = np.random.exponential(15.0, n_samples) + 5
    ch4 = np.random.exponential(20.0, n_samples) + 8
    c2h6 = np.random.exponential(12.0, n_samples) + 4
    c2h4 = np.random.exponential(18.0, n_samples) + 6
    c2h2 = np.random.exponential(1.5, n_samples) + 0.2
    co = np.random.normal(250.0, 50.0, n_samples)
    co2 = np.random.normal(2200.0, 300.0, n_samples)
    
    bdv = np.random.normal(65.0, 8.0, n_samples)        # Breakdown Voltage (kV)
    moisture = np.random.normal(14.0, 3.5, n_samples)   # ppm
    oil_temp = np.random.normal(55.0, 7.0, n_samples)   # deg C
    winding_temp = oil_temp + np.random.normal(12.0, 2.5, n_samples)
    age = np.random.uniform(2.0, 28.0, n_samples)
    
    # Inject fault specific gas signatures according to IEC 60599
    # 1. Partial Discharge (PD): High Hydrogen (H2)
    mask_pd = (fault_classes == 1)
    h2[mask_pd] += np.random.uniform(200, 800, sum(mask_pd))
    ch4[mask_pd] += np.random.uniform(30, 80, sum(mask_pd))
    
    # 2. Thermal T1/T2 (<700C): High Methane (CH4) and Ethane (C2H6)
    mask_t12 = (fault_classes == 2)
    ch4[mask_t12] += np.random.uniform(150, 450, sum(mask_t12))
    c2h6[mask_t12] += np.random.uniform(100, 300, sum(mask_t12))
    winding_temp[mask_t12] += np.random.uniform(20, 35, sum(mask_t12))
    
    # 3. Thermal T3 (>700C): Dominant Ethylene (C2H4)
    mask_t3 = (fault_classes == 3)
    c2h4[mask_t3] += np.random.uniform(350, 1100, sum(mask_t3))
    ch4[mask_t3] += np.random.uniform(100, 250, sum(mask_t3))
    oil_temp[mask_t3] += np.random.uniform(25, 45, sum(mask_t3))
    winding_temp[mask_t3] += np.random.uniform(40, 70, sum(mask_t3))
    
    # 4. Electrical Arcing (D1/D2): High Acetylene (C2H2)
    mask_arc = (fault_classes == 4)
    c2h2[mask_arc] += np.random.uniform(80, 500, sum(mask_arc))
    h2[mask_arc] += np.random.uniform(150, 600, sum(mask_arc))
    bdv[mask_arc] -= np.random.uniform(25, 45, sum(mask_arc))
    
    # Compute Continuous Health Index (100 = Perfect, 0 = Catastrophic)
    tcg_raw = h2 + ch4 + c2h6 + c2h4 + c2h2 + co
    health_index = 100.0 - (
        (tcg_raw / 35.0) +
        ((70.0 - np.clip(bdv, 15, 70)) * 0.8) +
        (moisture * 0.7) +
        (age * 0.6) +
        ((winding_temp - 60.0).clip(0, 100) * 0.4)
    )
    health_index = np.clip(health_index + np.random.normal(0, 3, n_samples), 5, 99)
    
    fault_labels = {
        0: "Nominal_Normal",
        1: "Partial_Discharge",
        2: "Thermal_Fault_T1_T2",
        3: "Thermal_Fault_T3",
        4: "Electrical_Arcing"
    }
    
    df = pd.DataFrame({
        "H2(ppm)": np.round(h2, 1),
        "CH4(ppm)": np.round(ch4, 1),
        "C2H6(ppm)": np.round(c2h6, 1),
        "C2H4(ppm)": np.round(c2h4, 1),
        "C2H2(ppm)": np.round(c2h2, 2),
        "CO(ppm)": np.round(co, 1),
        "CO2(ppm)": np.round(co2, 1),
        "BDV(kV)": np.round(bdv, 1),
        "Moisture(ppm)": np.round(moisture, 1),
        "Oil_Temp(C)": np.round(oil_temp, 1),
        "Winding_Temp(C)": np.round(winding_temp, 1),
        "Age(Years)": np.round(age, 1),
        "Health_Index": np.round(health_index, 1),
        "Fault_Class": [fault_labels[c] for c in fault_classes]
    })
    
    df.to_csv(path, index=False)
    print(f"Transformer dataset cached at: {path}")
    return df

df_transformer = load_or_generate_transformer_dataset(DATASET_PATH)
df_transformer.head()

Loading transformer dataset from: data/transformer_failures.csv


## 3. Power Transformer Diagnostic Ratio Engineering
According to IEEE C57.104 & IEC 60599:
1. **Total Combustible Gas (TCG):**
   $$\text{TCG} = H_2 + CH_4 + C_2H_6 + C_2H_4 + C_2H_2 + CO$$
2. **Duval Triangle Coordinates:**
   - $\%CH_4 = \frac{CH_4}{CH_4 + C_2H_4 + C_2H_2} \times 100$
   - $\%C_2H_4 = \frac{C_2H_4}{CH_4 + C_2H_4 + C_2H_2} \times 100$
   - $\%C_2H_2 = \frac{C_2H_2}{CH_4 + C_2H_4 + C_2H_2} \times 100$
3. **Rogers Ratios:**
   - $R_1 = CH_4 / (H_2 + 1e-4)$
   - $R_2 = C_2H_2 / (C_2H_4 + 1e-4)$
   - $R_3 = C_2H_4 / (C_2H_6 + 1e-4)$
4. **Thermal Gradient:** $\Delta T = T_{\text{winding}} - T_{\text{oil}}$

In [6]:
def engineer_transformer_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    
    # 1. Total Combustible Gas
    data['TCG(ppm)'] = data['H2(ppm)'] + data['CH4(ppm)'] + data['C2H6(ppm)'] + data['C2H4(ppm)'] + data['C2H2(ppm)'] + data['CO(ppm)']
    
    # 2. Duval Triangle Denominator
    duval_sum = data['CH4(ppm)'] + data['C2H4(ppm)'] + data['C2H2(ppm)'] + 1e-5
    data['Duval_CH4_%'] = (data['CH4(ppm)'] / duval_sum) * 100.0
    data['Duval_C2H4_%'] = (data['C2H4(ppm)'] / duval_sum) * 100.0
    data['Duval_C2H2_%'] = (data['C2H2(ppm)'] / duval_sum) * 100.0
    
    # 3. Rogers Ratios
    data['Rogers_R1_CH4_H2'] = data['CH4(ppm)'] / (data['H2(ppm)'] + 1e-4)
    data['Rogers_R2_C2H2_C2H4'] = data['C2H2(ppm)'] / (data['C2H4(ppm)'] + 1e-4)
    data['Rogers_R3_C2H4_C2H6'] = data['C2H4(ppm)'] / (data['C2H6(ppm)'] + 1e-4)
    
    # 4. Thermal Gradient
    data['Delta_Temp_Winding_Oil'] = data['Winding_Temp(C)'] - data['Oil_Temp(C)']
    
    # 5. Dielectric Degradation Index
    data['Dielectric_Degradation_Index'] = data['Moisture(ppm)'] / (data['BDV(kV)'] + 0.1)
    
    # 6. CO2 to CO ratio (Cellulose breakdown)
    data['CO2_CO_Ratio'] = data['CO2(ppm)'] / (data['CO(ppm)'] + 1e-4)
    
    return data

df_tx_eng = engineer_transformer_features(df_transformer)
print(f"Features after engineering: {df_tx_eng.shape[1]}")

Features after engineering: 24


## 4. Multi-Class Fault Classification (XGBoost)
We map the fault strings to categorical codes and train an XGBoost multi-class classifier.

In [8]:
label_enc = LabelEncoder()
y_class = label_enc.fit_transform(df_tx_eng['Fault_Class'])

X_tx = df_tx_eng.drop(columns=['Fault_Class', 'Health_Index'])
y_health = df_tx_eng['Health_Index']

X_train, X_test, y_train_cls, y_test_cls, y_train_reg, y_test_reg = train_test_split(
    X_tx, y_class, y_health, test_size=0.20, random_state=42, stratify=y_class
)

# 1. Multi-class Fault Classifier
xgb_classifier = xgb.XGBClassifier(
    n_estimators=180,
    max_depth=5,
    learning_rate=0.06,
    objective='multi:softprob',
    num_class=len(label_enc.classes_),
    random_state=42,
    eval_metric='mlogloss'
)
xgb_classifier.fit(X_train, y_train_cls)

# 2. Continuous Health Index Regressor
lgb_regressor = lgb.LGBMRegressor(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.05,
    random_state=42,
    verbose=-1
)
lgb_regressor.fit(X_train, y_train_reg)

# Evaluate Classification
y_pred_cls = xgb_classifier.predict(X_test)
cls_acc = accuracy_score(y_test_cls, y_pred_cls)

# Evaluate Regression
y_pred_reg = lgb_regressor.predict(X_test)
reg_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
reg_r2 = r2_score(y_test_reg, y_pred_reg)

print(f"Transformer Fault Classifier Accuracy: {cls_acc * 100:.2f}%")
print(f"Health Index Regressor RMSE: {reg_rmse:.2f} | R2 Score: {reg_r2:.4f}")
print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test_cls, y_pred_cls, target_names=label_enc.classes_))

Transformer Fault Classifier Accuracy: 96.81%
Health Index Regressor RMSE: 3.20 | R2 Score: 0.9673

=== CLASSIFICATION REPORT ===
                       precision    recall  f1-score   support

Cellulose_Degradation       0.98      0.98      0.98        49
    Electrical_Arcing       0.86      1.00      0.92         6
       Nominal_Normal       1.00      1.00      1.00        12
    Partial_Discharge       0.67      1.00      0.80         2
  Thermal_Fault_T1_T2       1.00      0.96      0.98        23
     Thermal_Fault_T3       1.00      0.50      0.67         2

             accuracy                           0.97        94
            macro avg       0.92      0.91      0.89        94
         weighted avg       0.97      0.97      0.97        94



## 5. Confusion Matrix & Health Index Calibration

In [10]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Classification Confusion Matrix
cm = confusion_matrix(y_test_cls, y_pred_cls)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd', ax=ax1,
            xticklabels=label_enc.classes_, yticklabels=label_enc.classes_)
ax1.set_title('Transformer Failure Mode Classification')
ax1.set_xticklabels(label_enc.classes_, rotation=30, ha='right')

# Regression Health Index Calibration
ax2.scatter(y_test_reg, y_pred_reg, alpha=0.6, color='#10b981', edgecolors='k', s=40)
ax2.plot([0, 100], [0, 100], 'r--', lw=2, label='Perfect Calibration')
ax2.set_xlabel('Actual Health Index')
ax2.set_ylabel('Predicted Health Index')
ax2.set_title(f'Health Index Calibration (R² = {reg_r2:.3f})')
ax2.legend()

plt.tight_layout()
plt.show()

## 6. TreeSHAP Explainability for Transformer Diagnostics
Shows which specific dissolved gas drove the diagnosis (e.g., Acetylene driving Arcing).

In [12]:
explainer = shap.TreeExplainer(xgb_classifier)
shap_values = explainer.shap_values(X_test)

# Top features globally
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, class_names=label_enc.classes_, plot_type="bar", show=False)
plt.title("Power Transformer: Driving Gas & Thermal Attributions (SHAP)", fontsize=13)
plt.tight_layout()
plt.show()

## 7. Exporting Transformer Diagnostic Artifacts

In [14]:
artifacts = {
    "classifier": xgb_classifier,
    "regressor": lgb_regressor,
    "label_encoder": label_enc,
    "feature_names": list(X_tx.columns),
    "classes": list(label_enc.classes_),
    "version": "1.0.0-siemens-transformer"
}

joblib.dump(artifacts, "models/transformer_health_model.joblib")
print("Saved artifacts to 'models/transformer_health_model.joblib' successfully!")

Saved artifacts to 'models/transformer_health_model.joblib' successfully!
